In [1]:
import pandas as pd
import torch
import json
import sys
from pathlib import Path

def find_project_root(marker=".git") -> Path:
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root (no {marker} found)")

project_root = find_project_root()
sys.path.append(str(project_root))

from src.tokenizer import char_tokenize, atom_tokenize, build_vocab, encode
from src.split import scaffold_split

In [2]:
df = pd.read_csv("../data/zinc250k.csv")
df.head()

,Unnamed: 0,index,smiles,selfies,logP,qed,SAS
0,0,0,CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1\n,[C][C][Branch1][C][C][Branch1][C][C][C][=C][C]...,5.05060,0.702012,2.084095
1,1,1,C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1\n,[C][C@@H1][C][C][Branch2][Ring1][Ring2][N][C][...,3.11370,0.928975,3.432004
2,2,2,N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...,[N][#C][C][=C][C][=C][Branch2][Ring2][Ring2][C...,4.96778,0.599682,2.470633
3,3,3,CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c...,[C][C][O][C][=Branch1][C][=O][C@@H1][C][C][C][...,4.00022,0.690944,2.822753
4,4,4,N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C([O-])[C@H](C#...,[N][#C][C][=C][Branch2][Ring1][Ring1][S][C][C]...,3.60956,0.789027,4.035182


Smiles strings end in '\n' token, use strip to remove it.

In [3]:
df['smiles'] = df['smiles'].str.strip()

In [4]:
token2idx, idx2token = build_vocab(df['smiles'].tolist(), char_tokenize)
print(token2idx)
print(idx2token)

{'<PAD>': 0, '<START>': 1, '<END>': 2, '#': 3, '(': 4, ')': 5, '+': 6, '-': 7, '/': 8, '1': 9, '2': 10, '3': 11, '4': 12, '5': 13, '6': 14, '7': 15, '8': 16, '=': 17, '@': 18, 'B': 19, 'C': 20, 'F': 21, 'H': 22, 'I': 23, 'N': 24, 'O': 25, 'P': 26, 'S': 27, '[': 28, '\\': 29, ']': 30, 'c': 31, 'l': 32, 'n': 33, 'o': 34, 'r': 35, 's': 36}
{0: '<PAD>', 1: '<START>', 2: '<END>', 3: '#', 4: '(', 5: ')', 6: '+', 7: '-', 8: '/', 9: '1', 10: '2', 11: '3', 12: '4', 13: '5', 14: '6', 15: '7', 16: '8', 17: '=', 18: '@', 19: 'B', 20: 'C', 21: 'F', 22: 'H', 23: 'I', 24: 'N', 25: 'O', 26: 'P', 27: 'S', 28: '[', 29: '\\', 30: ']', 31: 'c', 32: 'l', 33: 'n', 34: 'o', 35: 'r', 36: 's'}


In [5]:
training_set, holdout_set, holdout_scaffolds = scaffold_split(df['smiles'].tolist(), holdout_fraction=0.2, random_state=6)

No rings:  1109
Actual Holdout %:  0.20004008739051132


In [6]:
training_smiles = []
max_len = 69
dropped = 0
for smiles in training_set:
    encoded = encode(smiles, char_tokenize, token2idx, max_len)
    if encoded is not None:
        training_smiles.append(encoded)
    else:
        dropped += 1

print(f"Dropped {dropped} training molecules over max_len={max_len}")

holdout_smiles = []
dropped = 0
for smiles in holdout_set:
    encoded = encode(smiles, char_tokenize, token2idx, max_len)
    if encoded is not None:
        holdout_smiles.append(encoded)
    else:
        dropped += 1

print(f"Dropped {dropped} holdout molecules over max_len={max_len}")

training_tensor = torch.tensor(training_smiles, dtype=torch.long)
holdout_tensor = torch.tensor(holdout_smiles, dtype=torch.long)

Dropped 2351 training molecules over max_len=69
Dropped 632 holdout molecules over max_len=69


In [7]:
torch.save(training_tensor, '../data/processed/char_tokenized/training_tensor.pt')
torch.save(holdout_tensor, '../data/processed/char_tokenized/holdout_tensor.pt')

with open("../data/processed/char_tokenized/holdout_scaffolds.json", "w") as file:
    json.dump(list(holdout_scaffolds), file, indent=4)

metadata = {
    "token2idx": token2idx,
    "max_len": max_len,
    "strategy": "char_tokenize"
}

with open("../data/processed/char_tokenized/metadata.json", "w") as file:
    json.dump(metadata, file, indent=4)